<a href="https://colab.research.google.com/github/Data-Alia/1Project/blob/main/equity_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install yfinance to fetch live stock data if not already present
!pip install yfinance -q

import pandas as pd
import yfinance as yf

# Define our tickers
tickers = ["NVDA", "AVGO"]
data_summary = []

print("Fetching live market data from Yahoo Finance...")

for ticker in tickers:
  stock = yf.Ticker(ticker)
  info = stock.info

  # Extract key metrics safely
  name = info.get("longName", ticker)
  market_cap = info.get("marketCap", 0)
  beta = info.get("beta", 1.0)
  current_price = info.get(
      "currentPrice", info.get("regularMarketPrice", 0)
  )

  data_summary.append({
      "Company": name,
      "Ticker": ticker,
      "Current Price ($)": current_price,
      "Market Cap ($B)": round(market_cap / 1e9, 2),
      "Beta": beta,
  })

# Convert to a clean DataFrame and display
df = pd.DataFrame(data_summary)
display(df)

Fetching live market data from Yahoo Finance...


,Company,Ticker,Current Price ($),Market Cap ($B),Beta
0,NVIDIA Corporation,NVDA,226.32,5464.95,2.216
1,Broadcom Inc.,AVGO,354.73,1693.35,1.476


In [3]:
import numpy as np
import pandas as pd
import yfinance as yf


def run_automated_dcf(ticker, base_fcf, growth_rates, terminal_growth=0.025, risk_free_rate=0.042, market_risk_premium=0.055, tax_rate=0.21):
    stock = yf.Ticker(ticker)
    info = stock.info
    bs = stock.balance_sheet

    # 1. Cost of Capital (WACC) Calculation
    beta = info.get("beta", 1.1)
    if beta is None or pd.isna(beta):
        beta = 1.1
    cost_of_equity = risk_free_rate + (beta * market_risk_premium)

    market_cap = info.get("marketCap", 1e12)

    # Extract total debt safely
    total_debt = 0
    for row in ["Total Debt", "Long Term Debt"]:
        if row in bs.index:
            val = bs.loc[row].iloc[0]
            if pd.notna(val):
                total_debt = val
                break

    # Extract cash safely
    cash = 0
    for row in ["Cash And Cash Equivalents", "Cash Cash Equivalents And Short Term Investments"]:
        if row in bs.index:
            val = bs.loc[row].iloc[0]
            if pd.notna(val):
                cash = val
                break

    total_capital = market_cap + total_debt
    weight_equity = market_cap / total_capital
    weight_debt = total_debt / total_capital
    cost_of_debt = 0.055  # Proxy cost of debt

    wacc = (weight_equity * cost_of_equity) + (weight_debt * cost_of_debt * (1 - tax_rate))

    # 2. FCF Projections & Discounting
    discounted_fcfs = []
    current_fcf = base_fcf
    for i, g in enumerate(growth_rates):
        current_fcf *= (1 + g)
        pv_fcf = current_fcf / ((1 + wacc) ** (i + 1))
        discounted_fcfs.append(pv_fcf)

    sum_pv_fcfs = sum(discounted_fcfs)

    # 3. Terminal Value & Enterprise Value
    terminal_value = (current_fcf * (1 + terminal_growth)) / (wacc - terminal_growth)
    pv_terminal_value = terminal_value / ((1 + wacc) ** len(growth_rates))

    enterprise_value = sum_pv_fcfs + pv_terminal_value
    equity_value = enterprise_value + cash - total_debt
    shares_outstanding = info.get("sharesOutstanding", 1e9)
    implied_share_price = equity_value / shares_outstanding
    current_price = info.get("currentPrice", info.get("regularMarketPrice", 0))

    return {
        "Ticker": ticker,
        "Current Beta": round(beta, 2),
        "WACC (%)": round(wacc * 100, 2),
        "Enterprise Value ($B)": round(enterprise_value / 1e9, 2),
        "Equity Value ($B)": round(equity_value / 1e9, 2),
        "Implied Share Price ($)": round(implied_share_price, 2),
        "Market Share Price ($)": current_price
    }

# Run models with custom baseline assumptions for NVDA and AVGO
# (Using realistic base FCF scale estimates for demonstration)
nvda_result = run_automated_dcf("NVDA", base_fcf=60e9, growth_rates=[0.30, 0.25, 0.20, 0.15, 0.10])
avgo_result = run_automated_dcf("AVGO", base_fcf=25e9, growth_rates=[0.18, 0.15, 0.12, 0.10, 0.08])

# Combine into a final comparative report DataFrame
valuation_df = pd.DataFrame([nvda_result, avgo_result])
print("--- Automated DCF Comparative Valuation Model ---")
display(valuation_df)

--- Automated DCF Comparative Valuation Model ---


,Ticker,Current Beta,WACC (%),Enterprise Value ($B),Equity Value ($B),Implied Share Price ($),Market Share Price ($)
0,NVDA,2.22,16.36,868.95,868.52,35.97,226.32
1,AVGO,1.48,12.02,407.94,358.98,75.20,354.73
